# MyoChallenge 2026 — ChaseTag quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MyoHub/myosuite/blob/dev/tutorials/5.6_ChaseTag_MyoChallenge.ipynb)

For **first-time invitees**. About 10 minutes:

1. Register on the **website** (not in this notebook)
2. Copy your **CLI token** from Submissions
3. Export the [tutorial 5.4](https://colab.research.google.com/github/MyoHub/myosuite/blob/dev/tutorials/5.4_Directional_Locomotion.ipynb) walker (a myoSuperMimic-derived directional walker) to ONNX and submit it
4. Watch the result on the **website**
5. Optional: submit a stronger start (a self-play refinement of the myoSuperMimic DAgger student), or train your own walker in 5.4 and re-export
6. Optional: **play** `myoChallengeChaseTagFBVs-v0` locally (score + render)

Site: [mc26.myosuite.org](https://mc26.myosuite.org)

Invitation only. If the site says your email is not on the list, [apply here](https://docs.google.com/forms/d/e/1FAIpQLSf0tsCktz3adRUB6YGd0OfQMP9lQDrzeHeZCozrlVDN2J6_qw/viewform) first.

## What you submit

Each team submits **one file: a single player**.

You do **not** submit an opponent. The server always provides the other player.

| You send | The server does |
|---|---|
| **One** ONNX model | Runs it as player A |
| | Puts a champion (or another team) as player B |
| | Scores the match |

The file is a neural net that, every step:

- **reads** 546 numbers — your body, the opponent’s **hip** (not their muscles), and a **chase/evade flag**
- **writes** 354 numbers — your muscle activations, each between 0 and 1

```
obs (546)  →  YOUR MODEL  →  action (354)
```

You do **not** pick chase vs evade. One file plays both. The server assigns the role each episode and writes it into the observation. Opponent hip is only **where they are** — not whether to hunt or flee.

| Slice | What it is |
|---|---|
| `obs[0:537]` | Your body (joints, velocities, muscles, pelvis) |
| `obs[537:544]` | Opponent hip relative pose (where they are) |
| `obs[544:546]` | **Role** — `[1, 0]` = **chase** (go toward them), `[0, 1]` = **evade** (run away) |

Do not infer chase/evade from hip position. The same offset means opposite actions depending on those last two numbers. The export cell already reads `obs[544:546]` and flips heading.

That same file is ranked against a fixed champion during the cycle, then used in a 1v1 tournament against other teams.

## 1. Register on the website

Do this in the browser, not in Colab.

1. Open [mc26.myosuite.org](https://mc26.myosuite.org).
2. Click **Register your team**.
3. Fill **team name**, **email**, and **password** (8+ characters). Country and education are optional.
4. Click **Register team**. Already registered? Use **Sign in**.

<p><img alt="Register and Sign in on mc26.myosuite.org" width="480" src="https://raw.githubusercontent.com/MyoHub/myosuite/dev/tutorials/files/5.6/colab-auth.png"></p>

## 2. Copy your CLI token

On the website, open **Submissions** and copy the **CLI submission token**. Paste it with your email into the Colab cell below. Do not share the token.

<p><img alt="CLI submission token on Submissions" width="440" src="https://raw.githubusercontent.com/MyoHub/myosuite/dev/tutorials/files/5.6/colab-token.png"></p>

## 3. Export the 5.4 walker and submit it

The cells below take the tutorial 5.4 walker, wrap it for ChaseTag and upload the ONNX.

The walker is a student of the steered **myoSuperMimic** teacher (tutorial 5.7, `training/chasetag_teacher`) on
myo_sim's `myofullbody`, the body of the scored game. It reads a **528**-number directional observation: its body, and
a commanded heading **in its own yaw frame**. The server sends **546**. `export_chasetag_onnx`
(`chasetag_teacher.directional`) puts the 546 → 528 mapping inside the ONNX:

- the heading points at the opponent's hip, or away from it, depending on the **role flag** `obs[544:546]`
  (`[1, 0]` chase, `[0, 1]` evade);
- it turns off the ±6 m fence;
- it is rotated into the walker's yaw frame.

Do not export the raw 528-d net.

If you trained a walker in 5.4, upload its `walker_myofullbody.pt` into this Colab first; that file is used instead of
the download. Walkers of the earlier 5.4 (`policy_bc_best.pt`, `policy_heading_table.pt`, on the MuscleMimic body
with a world-frame heading) do not load here.

The walker and the players below come from `myohub/myosuite-3-baselines`; with `$MYOSUITE_BASELINES_DIR` set to a
local folder laid out like that repo, `download_baseline_file` reads them from there instead.

Paste email + token, then **Runtime → Run all**.

In [1]:
# @title Install
try:  # already installed (e.g. a local checkout)
    import chasetag_teacher.directional, evalmc, onnx, myosuite  # noqa: F401, E401
except ImportError:
    %pip -q install -U "evalmc>=0.0.4" onnx "myosuite @ git+https://github.com/MyoHub/myosuite.git@dev"
    # the walker export only (torch + MyoSuite): no training dependencies needed
    %pip -q install --no-deps "chasetag-teacher @ git+https://github.com/MyoHub/myosuite.git@dev#subdirectory=training/chasetag_teacher"
print("ok")

ok


In [2]:
# @title Paste email + CLI token from the website
EMAIL = ""  # @param {type:"string"}
AUTH_TOKEN = ""  # @param {type:"string"}

EMAIL = EMAIL.strip().lower()
AUTH_TOKEN = AUTH_TOKEN.strip()
SUBMIT = bool(EMAIL and AUTH_TOKEN)
if SUBMIT:
    print("email", EMAIL, "token length", len(AUTH_TOKEN))
else:
    print("No EMAIL / AUTH_TOKEN: the cells below export the player but do not submit it.")
    print("Paste both from mc26.myosuite.org → Submissions to submit.")

No EMAIL / AUTH_TOKEN: the cells below export the player but do not submit it.
Paste both from mc26.myosuite.org → Submissions to submit.


In [3]:
# @title Export the tutorial 5.4 walker → ChaseTag ONNX (546 → 354)
from pathlib import Path

from chasetag_teacher.directional import export_chasetag_onnx
from myosuite.core.hf_io import download_baseline_file

# Verified snapshot: tracker, directional walker and ChaseTag self-play player.
BASELINES_REVISION = "7ab693c740a9dd5a812ea5c1d6dcc52e2baa3e9b"

ONNX_PATH = Path("chasetag_player.onnx")
# Your own 5.4 walker if you uploaded one, else the published one.
WALKER = Path("walker_myofullbody.pt")
if not WALKER.is_file():
    WALKER = download_baseline_file("directional/walker_myofullbody.pt", revision=BASELINES_REVISION, fallback_to_main=False)
print("Walker:", Path(WALKER).name)

# 546-d FBVs observation → 528-d directional observation (heading toward the opponent, away from it as runner,
# in the walker's yaw frame) → walker → 354 muscle activations in [0, 1].
export_chasetag_onnx(WALKER, ONNX_PATH)
print(f"Wrote {ONNX_PATH}  (obs [batch, 546] → action [batch, 354])")

Walker: walker_myofullbody.pt
Wrote chasetag_player.onnx  (obs [batch, 546] → action [batch, 354])


In [4]:
# @title Submit
import subprocess
import sys

if not SUBMIT:
    print("Skipped: no EMAIL / AUTH_TOKEN.")
else:
    cmd = [
        sys.executable, "-m", "evalmc", "submit-onnx", str(ONNX_PATH),
        "-t", "chasetag", "-p", "1",
        "-e", EMAIL, "-a", AUTH_TOKEN,
    ]
    print("Uploading", ONNX_PATH.name, "…")
    proc = subprocess.run(cmd, capture_output=True, text=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
        raise SystemExit(proc.returncode)
    print("Upload accepted. Now open the website Submissions tab (next section).")

Skipped: no EMAIL / AUTH_TOKEN.


## 4. Watch evaluation on the website

Go back to [mc26.myosuite.org](https://mc26.myosuite.org) → **Submissions**. Refresh until the row moves **submitted → evaluating → scored**.

<p><img alt="Submissions table with status badges" width="440" src="https://raw.githubusercontent.com/MyoHub/myosuite/dev/tutorials/files/5.6/colab-watch.png"></p>

| Status | Meaning |
|---|---|
| **submitted** | queued |
| **evaluating** | running |
| **scored** | done — a number appears under Score |
| **failed** | click the row for the error |

This file is the 5.4 directional walker, not a random dummy. Section 5 has a stronger start; or train your own walker in 5.4, then re-run **Export** and **Submit**.

## 5. A stronger player

The upload above is the **tutorial 5.4 walker** steered by the role flag: it walks where the heading points, but it was
never trained in the game.

**A stronger start: a self-play refinement of the myoSuperMimic DAgger student.** `training/chasetag_teacher`
distils the teacher of [tutorial 5.7](./5.7_ChaseTag_Steered_Teacher.ipynb) into one 546 → 354 network by DAgger in
the scored game itself, both roles; this player refines that student with self-play. Its ONNX takes the raw
observation and can be submitted as is. The next cell downloads it into `ONNX_PATH`; re-run **Submit** afterwards.

The project's evaluation matrix (48 episodes per match and side, action noise 0.02) compares them with the shipped
FBVs baseline (`onnx/chasetag_fbvs_baseline.onnx`):

| Player | Tags a player that does nothing | Wins head-to-head vs the FBVs baseline |
|---|---:|---:|
| FBVs baseline | 100 % | 50 % |
| tutorial 5.4 walker (the upload above) | 100 % | 64 % |
| myoSuperMimic DAgger student | 98 % | 61 % |
| its self-play refinement (the stronger start) | 100 % | 76 % |

All of them leave plenty to win: they still fall in many bouts.

**Train your own.** Fine-tune any of them in the 1v1 game, train a walker in 5.4 and upload its `walker_myofullbody.pt`
here (then re-run **Export** and **Submit**), or distil your own player with `training/chasetag_teacher` (README).

[![Open tutorial 5.4 in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MyoHub/myosuite/blob/dev/tutorials/5.4_Directional_Locomotion.ipynb)

In [5]:
# @title Use the stronger start instead (then re-run Submit)
USE_STUDENT = False  # @param {type:"boolean"}

if USE_STUDENT:
    ONNX_PATH = download_baseline_file("onnx/chasetag_supermimic_selfplay.onnx", revision=BASELINES_REVISION, fallback_to_main=False)
    print("Player:", Path(ONNX_PATH).name, "— re-run Submit to upload it")
else:
    print("Keeping", ONNX_PATH.name, "(check USE_STUDENT to switch)")

Keeping chasetag_player.onnx (check USE_STUDENT to switch)


## 6. Play a local 1v1 and render a match (optional)

Plays your ONNX file as **both** players of `myoChallengeChaseTagFBVs-v0` (546-d) for a few episodes. Prints how each
ended (tag, fall, time) with player A's score. Episode 0 is recorded until it ends.

The game is built with its gait/steering shaping weights at 0: only the training reward changes (it then needs no
gait clip); observations and score are the scored game's.

Leave **Run** unchecked when you only want to submit (`Runtime → Run all`).

In [6]:
# @title Play myoChallengeChaseTagFBVs-v0 locally and record a match
import os

RUN = False  # @param {type:"boolean"}
EPISODES = 4  # @param {type:"integer"}
RUN = RUN or os.environ.get("MYOSUITE_RUN_1V1", "0") == "1"

if not RUN:
    print("Skipped. Check Run to play the 1v1 env locally and record a match.")
else:
    import subprocess
    import sys
    import warnings

    warnings.filterwarnings("ignore", category=UserWarning)
    if sys.platform != "darwin":
        os.environ.setdefault("MUJOCO_GL", "egl")
        os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
    if "google.colab" in sys.modules:
        subprocess.run(["apt-get", "-qq", "install", "-y", "libegl1", "libgles2"], check=False)
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "-U",
            "onnxruntime", "imageio", "imageio-ffmpeg",
        ])

    import imageio
    import mujoco
    import numpy as np
    import onnxruntime as ort
    from IPython.display import Video, display

    from myosuite import make_env
    from myosuite.envs.myo.tasks.challenge.chase_tag_vs.chase_tag_vs_task_config import (
        ChaseTagVsFullbodySteeredTaskConfig,
    )

    onnx_path = Path(ONNX_PATH)
    if not onnx_path.is_file():
        raise SystemExit(f"{onnx_path} not found: run the export cell or upload your .onnx first.")
    sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
    name = sess.get_inputs()[0].name

    def act(obs):
        x = np.asarray(obs, np.float32).reshape(1, -1)
        return np.clip(sess.run(None, {name: x})[0].reshape(-1), 0.0, 1.0)

    config = ChaseTagVsFullbodySteeredTaskConfig(w_gait_jpos=0.0, w_gait_jvel=0.0, w_steering=0.0)
    env = make_env("myoChallengeChaseTagFBVs-v0", task_config=config)
    raw = env.unwrapped

    renderer = mujoco.Renderer(raw.model, height=360, width=480)
    cam = mujoco.MjvCamera()
    cam.type = mujoco.mjtCamera.mjCAMERA_FREE
    cam.lookat[:] = [0.0, 0.0, 1.0]
    cam.distance = 5.0
    cam.azimuth = 135.0
    cam.elevation = -20.0

    frames = []
    print(f"{'ep':>3}  {'role':>5}  {'steps':>5}  {'t':>6}  {'end':<22}  {'score A':>7}")
    for episode in range(int(EPISODES)):
        obs, info = env.reset(seed=episode)
        n = 0
        while True:
            obs, _, terminated, truncated, info = env.step(
                {"agent_0": act(obs["agent_0"]), "agent_1": act(obs["agent_1"])}
            )
            n += 1
            if episode == 0:
                renderer.update_scene(raw.data, camera=cam)
                frames.append(np.asarray(renderer.render(), dtype=np.uint8))
            if terminated["agent_0"] or truncated["agent_0"]:
                break
        fell = [agent for agent, down in info["fell"].items() if down]
        end = "tag" if info["tagged"] else f"fall ({', '.join(fell)})" if fell else "time limit"
        print(f"{episode:3d}  {info['task']:5s}  {n:5d}  {info['elapsed_s']:5.2f}s  {end:<22}  {info['score']:7.3f}")
    renderer.close()
    env.close()

    out = Path("chasetag_fbvs.mp4")
    with imageio.get_writer(str(out), fps=33, quality=8, macro_block_size=1) as writer:
        for frame in frames[::3]:
            writer.append_data(frame)
    print("episode 0 recorded:", len(frames), "steps ->", out)
    display(Video(str(out), embed=True, html_attributes="controls loop"))

Skipped. Check Run to play the 1v1 env locally and record a match.
